# esa-lookup (self-contained notebook)

**Auto-generated** from `pipeline.py` + `excel_ops.py` + `sap_ops.py` by
`build_notebook.py`. To update after a code change: edit the .py file(s),
then run `py build_notebook.py`. Do not hand-edit this notebook.

Mass-lookup SAP data into an Excel workbook via transaction **ZTBV**,
plant **ESA1**. Two workflows:

- **TO** (3 steps): K → LTAP → M ; N+O → Z50CFG_ENG_CRNT → C/D/E + A + P ;
  C → Z50CFG_ENG_VALD → F..I
- **NOTIF** (2 steps): A → Z50CFG_ENG_CRNT → C/D/E ; C → Z50CFG_ENG_VALD
  → F..I + LID to J

## Prerequisites

- Windows + **SAP GUI for Windows** + **Microsoft Excel** (desktop)
- SAP GUI Scripting enabled (Options → Accessibility & Scripting)
- Python 3.10+ with `pip install pandas openpyxl pywin32`

## How to run

1. Log into SAP GUI (any session; the script attaches to the first one).
2. Cell → Run All. The config cell shows a **Browse** dialog to pick
   the workbook; Excel will auto-open it if it is not already open.

Per-run logs land in `%LOCALAPPDATA%\esa-lookup\logs\` (last 20 runs
retained).


## Imports

In [ ]:
from __future__ import annotations

import contextlib
import os
import re
import sys
import tempfile
import threading
import time
import traceback
from dataclasses import dataclass, field
from decimal import Decimal, InvalidOperation
from typing import Callable

import pandas as pd
import pythoncom
import win32com.client
from win32com.client import constants  # noqa: F401 (loaded lazily by pywin32)


## Excel COM helpers *(inlined from `excel_ops.py`)*

In [ ]:
XL_UP = -4162  # Excel: xlUp
XL_CALC_MANUAL = -4135
XL_CALC_AUTOMATIC = -4105


class ExcelError(RuntimeError):
    pass


@dataclass
class ExcelCtx:
    app: object
    book: object
    sheet: object


def excel_attach(path: str) -> ExcelCtx:
    """Return an ExcelCtx for `path`. If Excel already has the file open,
    reuse it; otherwise start Excel and open it read/write.
    """
    if not os.path.exists(path):
        raise ExcelError(f"Excel file not found:\n{path}")

    abs_path = os.path.abspath(path).lower()

    app = None
    try:
        app = win32com.client.GetActiveObject("Excel.Application")
    except Exception:
        app = win32com.client.Dispatch("Excel.Application")
        app.Visible = True

    # Fix I: OneDrive/SharePoint-hosted workbooks report FullName as an
    # https:// URL, not a local filesystem path. os.path.abspath on a URL
    # produces gibberish and never matches -- so we then Open() a second
    # (read-only) copy and abort. Match by URL-tail filename in that case.
    target_basename = os.path.basename(path).lower()
    book = None
    for i in range(1, app.Workbooks.Count + 1):
        b = app.Workbooks(i)
        try:
            fn = str(b.FullName)
        except Exception:
            continue
        fn_lower = fn.lower()
        if fn_lower.startswith(("http:", "https:")):
            tail = fn.rsplit("/", 1)[-1].lower()
            if tail == target_basename:
                book = b
                break
        else:
            try:
                if os.path.abspath(fn).lower() == abs_path:
                    book = b
                    break
            except Exception:
                continue

    # Fix 7: always pass an absolute path -- Excel resolves relative paths
    # against its own cwd, which is usually not what the caller wants.
    if book is None:
        book = app.Workbooks.Open(os.path.abspath(path), 0, False)

    if book.ReadOnly:
        raise ExcelError(
            "Workbook is open Read-Only. Close it in Excel or check "
            "OneDrive/SharePoint permissions, then retry.\n" + path
        )

    return ExcelCtx(app=app, book=book, sheet=book.Worksheets(1))


def last_row_in_column(sheet, col_index: int) -> int:
    return int(sheet.Cells(sheet.Rows.Count, col_index).End(XL_UP).Row)


def read_range_2d(sheet, range_str: str) -> list[list]:
    """Read a range in one COM call; normalize to a list-of-lists.

    Single-cell ranges come back as a scalar; single-row/single-column
    ranges come back as a 1-tuple of tuples in some cases -- normalize both.
    """
    raw = sheet.Range(range_str).Value
    if raw is None:
        return []
    if not isinstance(raw, tuple):
        return [[raw]]
    # Range with a single row: raw is a tuple of scalars
    if raw and not isinstance(raw[0], tuple):
        return [list(raw)]
    return [list(r) for r in raw]


def write_range_2d(sheet, range_str: str, values_2d: list[list]) -> None:
    """Write a 2D python list to a range in one COM call.

    Excel's COM interface accepts a tuple-of-tuples on the RHS of Range.Value.
    Fix J: pre-check for merged cells so a mid-write failure can never leave
    the sheet in a half-updated state.
    """
    if not values_2d:
        return
    rng = sheet.Range(range_str)
    try:
        merged = bool(rng.MergeCells)
    except Exception:
        # `MergeCells` returns None (not a bool) for a mixed-merge range;
        # that itself signals merged content is present.
        merged = True
    if merged:
        raise ExcelError(
            f"Cannot write to range {range_str}: it contains merged cells. "
            f"Unmerge those columns in Excel and re-run. (SAP lookup output "
            f"columns must be plain, unmerged cells.)"
        )
    payload = tuple(tuple(row) for row in values_2d)
    rng.Value = payload


def clear_range(sheet, range_str: str) -> None:
    sheet.Range(range_str).ClearContents()


def set_column_format_text(sheet, columns: str) -> None:
    """columns like 'A:E' or 'M:M'."""
    sheet.Range(f"{columns}").NumberFormat = "@"


def autofit(sheet, columns: str) -> None:
    sheet.Columns(columns).AutoFit()


@contextlib.contextmanager
def bulk_write(app):
    """Disable Excel screen updating and calc while writing large blocks."""
    prev_updating = app.ScreenUpdating
    prev_events = app.EnableEvents
    prev_calc = app.Calculation
    try:
        app.ScreenUpdating = False
        app.EnableEvents = False
        app.Calculation = XL_CALC_MANUAL
        yield
    finally:
        app.Calculation = prev_calc
        app.EnableEvents = prev_events
        app.ScreenUpdating = prev_updating


def stage_values_on_clipboard(app, values: list[str]) -> object:
    """Create a scratch workbook, dump `values` into Column A as text,
    Copy() them onto the OS clipboard, and return the scratch workbook so
    the caller can close it after SAP has consumed the paste.
    """
    scratch = app.Workbooks.Add()
    ws = scratch.Worksheets(1)
    ws.Columns("A").NumberFormat = "@"
    # Fix 1: one bulk COM assignment instead of N per-cell writes. On a
    # 5000-key input this is the difference between ~30 s and <1 s.
    if values:
        payload = tuple(("" if v is None else str(v),) for v in values)
        ws.Range(f"A1:A{len(values)}").Value = payload
        ws.Range(f"A1:A{len(values)}").Copy()
    return scratch


def close_scratch(scratch) -> None:
    try:
        scratch.Close(SaveChanges=False)
    except Exception:
        pass


def excel_save(book) -> None:
    """Save the workbook. Fix L: raise ExcelError on failure instead of
    silently swallowing it, so the pipeline can log a warning and the user
    knows to save manually rather than assuming success.
    """
    try:
        book.Save()
    except Exception as e:
        raise ExcelError(
            f"Excel refused to save the workbook: {e}. Data is written but "
            f"unsaved -- press Ctrl+S in Excel to persist it."
        ) from e


## SAP GUI scripting helpers *(inlined from `sap_ops.py`)*

In [ ]:
PLANT = "ESA1"
TRANSACTION = "ZTBV"


class SapError(RuntimeError):
    pass


@dataclass
class SapSession:
    session: object

    def find(self, oid):
        return self.session.findById(oid)


def sap_attach() -> SapSession:
    """Attach to the first running SAP GUI session. Raise SapError if none."""
    try:
        sap_gui = win32com.client.GetObject("SAPGUI")
    except Exception as e:
        raise SapError(
            "Cannot reach SAP GUI. Log into SAP GUI first and confirm "
            "'Enable scripting' is on (Options -> Accessibility & Scripting)."
        ) from e
    try:
        app = sap_gui.GetScriptingEngine
        conn = app.Children(0)
        sess = conn.Children(0)
    except Exception as e:
        raise SapError(
            "SAP GUI is running but no active session was found. Open a "
            "connection and log in, then retry."
        ) from e
    return SapSession(sess)


def close_lingering_modals(s: SapSession, log=None) -> int:
    """Fix K: close any wnd[1..N] popups left over from a previous run.
    Typing an okcd on wnd[0] while a modal is open is silently ignored,
    which then causes the next few button-presses to hit the WRONG screen.
    Returns the number of modals actually closed (for diagnostic logging).
    """
    closed = 0
    for i in range(9, 0, -1):
        try:
            title = ""
            try:
                title = str(s.find(f"wnd[{i}]").Text or "")
            except Exception:
                pass
            s.find(f"wnd[{i}]").close()
            closed += 1
            if log:
                log(f"SAP: closed leftover modal wnd[{i}]" +
                    (f" (title: {title!r})" if title else ""))
        except Exception:
            pass
    return closed


def open_ztbv_table(s: SapSession, table: str, log=None) -> None:
    """Navigate to /nZTBV -> plant + table -> F8 (enter table view)."""
    if log:
        log(f"SAP: /n{TRANSACTION} -> {table} @ plant {PLANT}")
    n_closed = close_lingering_modals(s, log=log)
    if n_closed and log:
        log(f"SAP: {n_closed} lingering modal(s) closed before navigation")
    s.find("wnd[0]").maximize()
    s.find("wnd[0]/tbar[0]/okcd").Text = f"/n{TRANSACTION}"
    s.find("wnd[0]").sendVKey(0)
    time.sleep(0.3)
    s.find("wnd[0]/usr/txtD_WERKS").Text = PLANT
    s.find("wnd[0]/usr/ctxtD_TAB").Text = table
    s.find("wnd[0]/usr/ctxtD_TAB").SetFocus()
    s.find("wnd[0]/usr/ctxtD_TAB").caretPosition = len(table)
    s.find("wnd[0]/tbar[1]/btn[8]").press()  # F8 -> selection screen
    time.sleep(0.3)


def paste_multi_value_filter(
    s: SapSession, push_button_id: str, values: list[str], log=None
) -> None:
    """Click multi-select push button, upload clipboard, OK.

    Caller is responsible for having already staged `values` on the OS
    clipboard (typically via Excel: put values in Column A of a temp
    workbook then .Copy()). This mirrors the notebook's approach.
    """
    if log:
        log(f"SAP: pasting {len(values)} filter values via clipboard")
    s.find(push_button_id).press()
    time.sleep(0.3)
    # In the multi-value dialog: btn[24] = Upload from Clipboard, btn[8] = OK
    s.find("wnd[1]/tbar[0]/btn[24]").press()
    time.sleep(0.3)
    s.find("wnd[1]/tbar[0]/btn[8]").press()
    time.sleep(0.2)


def execute_query(s: SapSession, log=None) -> None:
    """Press F8 on the selection screen to run the query."""
    if log:
        log("SAP: executing query (F8)")
    s.find("wnd[0]/tbar[1]/btn[8]").press()
    time.sleep(0.5)


def export_alv_to_file(
    s: SapSession, target_dir: str, filename: str, log=None, timeout_s: int = 30
) -> str:
    """Export the current ALV grid to a spreadsheet file in `target_dir`.

    Tries the modern `&XXL` toolbar path first, then falls back to `&PC`
    (Save as Local File) with the spreadsheet format radio.

    Returns the full path to the exported file. Raises SapError on failure
    or timeout.
    """
    os.makedirs(target_dir, exist_ok=True)
    target_path = os.path.join(target_dir, filename)
    # Fix A: os.remove can silently fail (AV / lingering handle). Even if the
    # stale file survives, we only accept files whose mtime is newer than the
    # moment we triggered the export, so we can never return a prior step's
    # data as the current step's result.
    if os.path.exists(target_path):
        try:
            os.remove(target_path)
        except OSError:
            pass
    export_started_at = time.time()

    grid = s.find("wnd[0]/shellcont/shell")

    last_err: Exception | None = None
    for approach in ("XXL", "PC"):
        try:
            if approach == "XXL":
                if log:
                    log("SAP: exporting via &MB_EXPORT / &XXL")
                grid.pressToolbarContextButton("&MB_EXPORT")
                time.sleep(0.2)
                grid.selectContextMenuItem("&XXL")
            else:
                if log:
                    log("SAP: retrying export via &PC")
                grid.pressToolbarContextButton("&MB_EXPORT")
                time.sleep(0.2)
                grid.selectContextMenuItem("&PC")
                time.sleep(0.4)
                # Format picker -- select spreadsheet radio if present
                try:
                    s.find(
                        "wnd[1]/usr/subSUBSCREEN_STEPLOOP:SAPLSPO5:0150/"
                        "radSPOPLI-SELFLAG[1,0]"
                    ).Select()
                except Exception:
                    pass
                try:
                    s.find("wnd[1]/tbar[0]/btn[0]").press()
                except Exception:
                    pass
            time.sleep(0.5)
            # Save-As dialog: DY_PATH + DY_FILENAME + Save
            s.find("wnd[1]/usr/ctxtDY_PATH").Text = target_dir
            s.find("wnd[1]/usr/ctxtDY_FILENAME").Text = filename
            # btn[11] = "Replace" / Save on standard SAP save-as dialog
            s.find("wnd[1]/tbar[0]/btn[11]").press()
            # Wait for file, refusing any pre-existing stale copy (Fix A).
            deadline = time.time() + timeout_s
            while time.time() < deadline:
                if os.path.exists(target_path) and os.path.getsize(target_path) > 0:
                    try:
                        fresh = os.path.getmtime(target_path) >= export_started_at
                    except OSError:
                        fresh = False
                    if fresh:
                        if log:
                            log(f"SAP: export saved -> {target_path}")
                        return target_path
                time.sleep(0.25)
            raise SapError(f"Export timed out (>{timeout_s}s) waiting for {target_path}")
        except Exception as e:
            last_err = e
            if log:
                log(f"SAP: {approach} export attempt failed: "
                    f"{type(e).__name__}: {e}")
            # Best-effort: close any modal that might be hanging around so
            # the fallback attempt (or a subsequent step) can navigate.
            try:
                s.find("wnd[1]/tbar[0]/btn[12]").press()  # Cancel
                if log:
                    log("SAP: cancelled leftover modal to prepare fallback")
            except Exception:
                pass
            time.sleep(0.3)
            continue

    raise SapError(
        "ALV export failed via both &XXL and &PC. The exact save-dialog "
        "field IDs may differ on this SAP GUI version -- record one export "
        "via the SAP GUI Script Recorder and adjust `export_alv_to_file`."
    ) from last_err


# Multi-value push button IDs on the ZTBV selection screen per (table, field).
# These match the notebook. If a site uses a customized ZTBV layout the S<n>
# indices may shift -- record once with the Script Recorder and update here.
PUSH_BUTTONS = {
    ("LTAP", "TO_NUMBER"): "wnd[0]/usr/btn%_S3_%_APP_%-VALU_PUSH",
    ("Z50CFG_ENG_CRNT", "RSNUM"): "wnd[0]/usr/btn%_S15_%_APP_%-VALU_PUSH",
    ("Z50CFG_ENG_CRNT", "QMNUM"): "wnd[0]/usr/btn%_S29_%_APP_%-VALU_PUSH",
    ("Z50CFG_ENG_VALD", "OBJNR"): "wnd[0]/usr/btn%_S2_%_APP_%-VALU_PUSH",
}


## Pipeline core *(inlined from `pipeline.py`)*

Key normalization, workflow definitions, per-step orchestrator, and the
per-run file-log subsystem. All 10 notebook-parity fixes present.


In [ ]:
class Cancelled(Exception):
    """Raised inside a worker step when the user has pressed Stop."""


# ---------------------------------------------------------------------------
# Key normalization (ported from the notebook -- same rules, single copy)
# ---------------------------------------------------------------------------

# Fix E: only expand scientific notation when the whole string matches it,
# not any string that happens to contain the letter 'E'. Otherwise object
# numbers like "1E2000" get mangled to "100" + trailing garbage.
_SCI_NOTATION_RE = re.compile(r"^-?\d+(\.\d+)?[eE][+-]?\d+$")

# Rows whose primary key cell equals one of these (case-insensitive) are
# not sent to SAP as filter values, mirroring the notebook's paste-side
# guard: `if v != "" and v.upper() not in {"NOT FOUND","NOTFOUND"}`.
# Note: the notebook's WRITE-BACK loop still clears their output cells (via
# ClearContents + the non-match else branch), so this refactor does too --
# skip rows are treated as non-matches for the write path, not preserved.
_SKIP_KEY_MARKERS = frozenset({"NOT FOUND", "NOTFOUND"})


def _clean_cell(value) -> str:
    """Turn a raw Excel/pandas value into a stripped string, treating None,
    NaN, and pywintypes cell-error ints as empty.
    - Fix B: str(float('nan')) is 'nan' -- filter NaN before it becomes a key.
    - Fix G: #N/A / #REF! / #VALUE! come back from Excel COM as large-negative
      int codes (e.g. -2146826281). Treat those as blanks too.
    """
    if value is None:
        return ""
    if isinstance(value, float):
        # NaN check without importing math.isnan (works for float NaN)
        if value != value:
            return ""
    if isinstance(value, int) and not isinstance(value, bool) and value < -2_000_000_000:
        return ""
    return str(value).strip()


def _canonicalize_number_shape(txt: str) -> str:
    """Shared body of normalize_key / clean_numeric_for_sap."""
    for ch in (" ", "\t", chr(160), "'", ","):
        txt = txt.replace(ch, "")
    if _SCI_NOTATION_RE.match(txt):
        try:
            txt = format(Decimal(txt), "f")
        except InvalidOperation:
            pass
    if "." in txt:
        left, right = txt.split(".", 1)
        if right == "" or set(right) <= {"0"}:
            txt = left
    return txt


def normalize_key(value) -> str:
    """Aggressively normalize an Excel value so it matches SAP's key form.

    - drop None, NaN, and Excel cell-error codes
    - strip whitespace / NBSP / apostrophes / commas
    - collapse scientific notation ONLY when the whole string is sci-notation
    - drop trailing '.0' / '.00' / '.'
    - drop leading zeros (keep at least one char)

    The notebook applied different (weaker) rules per step, but every rule
    used here is applied identically to both the Excel side and the SAP side
    inside `_build_lookup`, so this can only add matches, never subtract.
    """
    txt = _clean_cell(value)
    if not txt:
        return ""
    txt = _canonicalize_number_shape(txt)
    while len(txt) > 1 and txt.startswith("0"):
        txt = txt[1:]
    return txt


def clean_numeric_for_sap(value) -> str:
    """Like normalize_key but preserves leading zeros -- used when pasting
    numeric identifiers into SAP where SAP itself will canonicalize them.
    """
    txt = _clean_cell(value)
    if not txt:
        return ""
    return _canonicalize_number_shape(txt)


def _is_skip_key_cell(value) -> bool:
    """True when this Excel key cell should NOT be pasted into SAP's filter
    dialog (blank, or one of the 'already resolved' markers). Matches the
    notebook's paste-side guard. Skipped rows are still processed by the
    write-back loop as non-matches.
    """
    txt = _clean_cell(value)
    if not txt:
        return True
    return txt.upper() in _SKIP_KEY_MARKERS


# ---------------------------------------------------------------------------
# Workflow definitions
# ---------------------------------------------------------------------------

@dataclass
class ExtraOutput:
    """A per-step output column that lives outside the main output block.

    `sap_col=None` means "no SAP source" -- the column is only cleared on a
    non-match (used to mirror the notebook's TO Step 3 behavior on column J,
    which is cleared when a row fails to match but preserved when it does).

    `preserve_on_nonmatch=True` means "on non-match, do not touch this cell"
    (mirrors TO Step 2's column A: notebook comment "Do not touch Column A
    if there is no match").

    Skip rows (blank / "NOT FOUND" key) are treated identically to non-match
    rows for the extras, matching the notebook where the write-back loop
    iterates every row and only the key check inside the loop distinguishes
    match vs. else (skip rows fall into the else branch).
    """
    excel_col: int
    sap_col: str | None = None
    preserve_on_nonmatch: bool = False


@dataclass
class LookupStep:
    name: str                     # human label for logs
    sap_table: str                # e.g. "LTAP"
    push_button_field: str        # e.g. "TO_NUMBER" -- indexes PUSH_BUTTONS
    key_columns: list[int]        # 1-based Excel column indices used as key
    key_joiner: str = "|"         # how multi-column keys are joined
    sap_key_columns: list[str] = field(default_factory=list)     # SAP df cols composing the match key
    sap_output_columns: list[str] = field(default_factory=list)  # SAP df cols we copy to Excel
    excel_output_columns: list[int] = field(default_factory=list)  # 1-based Excel columns to write
    extras: list[ExtraOutput] = field(default_factory=list)      # per-cell exceptions to the main block
    # If set, write header + composite match key (normalize_key of every part
    # joined by key_joiner) to this 1-based column, rows 1..last_row.
    # Notebook TO-2 does this for column P as an audit trail.
    match_key_column: int | None = None
    match_key_header: str = "Excel Match Key Used"


WORKFLOWS = {
    "TO": [
        LookupStep(
            name="LTAP -> Unloading Point",
            sap_table="LTAP",
            push_button_field="TO_NUMBER",
            key_columns=[11],                     # K
            sap_key_columns=["TANUM"],            # LTAP TO number field
            sap_output_columns=["ABLAD"],
            excel_output_columns=[13],            # M
        ),
        LookupStep(
            name="Z50CFG_ENG_CRNT (Reservation) -> QMNUM/OBJNR/DISP",
            sap_table="Z50CFG_ENG_CRNT",
            push_button_field="RSNUM",
            key_columns=[14, 15],                 # N | O
            sap_key_columns=["RSNUM", "RSPOS"],
            sap_output_columns=["OBJNR", "DISP_MATNR", "DISP_QTY"],
            excel_output_columns=[3, 4, 5],       # C, D, E
            # Notebook: A gets written on match with QMNUM but MUST NOT be
            # touched on non-match ("Do not touch Column A if there is no
            # match" -- line 874 in the original notebook). ClearContents
            # covers C..E only, so A is preserved on skip as well.
            extras=[ExtraOutput(excel_col=1, sap_col="QMNUM", preserve_on_nonmatch=True)],
            match_key_column=16,                  # P: audit-trail composite key (notebook line 851, 863)
        ),
        LookupStep(
            name="Z50CFG_ENG_VALD -> Section/Module/Description/SalesDoc",
            sap_table="Z50CFG_ENG_VALD",
            push_button_field="OBJNR",
            key_columns=[3],                      # C
            sap_key_columns=["OBJNR"],
            # Notebook TO-3 sap_data_map only carries these 4 (LID is NOT
            # populated on match). Column J is cleared in the non-match
            # branch and left alone on match -- modeled by the extras entry.
            sap_output_columns=["Z_SECTION", "Z_MODULE", "DESCRIPT", "SALES_ORDER"],
            excel_output_columns=[6, 7, 8, 9],    # F..I
            extras=[ExtraOutput(excel_col=10)],   # J: preserve on match, clear on non-match/skip
        ),
    ],
    "NOTIF": [
        LookupStep(
            name="Z50CFG_ENG_CRNT (Notification) -> OBJNR/DISP_MATNR/DISP_QTY",
            sap_table="Z50CFG_ENG_CRNT",
            push_button_field="QMNUM",
            key_columns=[1],                      # A
            sap_key_columns=["QMNUM"],
            sap_output_columns=["OBJNR", "DISP_MATNR", "DISP_QTY"],
            excel_output_columns=[3, 4, 5],       # C, D, E
        ),
        LookupStep(
            name="Z50CFG_ENG_VALD -> Section/Module/Description/SalesDoc/LID",
            sap_table="Z50CFG_ENG_VALD",
            push_button_field="OBJNR",
            key_columns=[3],                      # C
            sap_key_columns=["OBJNR"],
            # Notebook NOTIF-2 fetches LID from SAP but never writes it (bug
            # -- match branch writes only F..I). The completion message and
            # this project's README both say LID -> J, so we treat that as
            # the user's true intent and route LID through an ExtraOutput.
            # ClearContents range therefore matches notebook (F..I only).
            sap_output_columns=["Z_SECTION", "Z_MODULE", "DESCRIPT", "SALES_ORDER"],
            excel_output_columns=[6, 7, 8, 9],    # F..I
            extras=[ExtraOutput(excel_col=10, sap_col="LID", preserve_on_nonmatch=False)],
        ),
    ],
}


# ---------------------------------------------------------------------------
# Event callback protocol
# ---------------------------------------------------------------------------

# on_event(kind, payload):
#   ("log", (message: str, level: "info"|"ok"|"warn"|"error"))
#   ("status", message: str)
#   ("progress", fraction: float 0..1)
#   ("done", ok: bool)

EventFn = Callable[[str, object], None]


# ---------------------------------------------------------------------------
# Per-run file log -- persistent record so a mid-run crash can be diagnosed
# after the fact even if the GUI closed. Path is echoed into the GUI log at
# run start.
# ---------------------------------------------------------------------------
_LOG_FH = None       # file handle
_LOG_FILE = ""       # current log path


def _open_run_log() -> str:
    """Open a timestamped log file for this run. Never raises; returns the
    path (empty on failure). Also prunes older runs to keep at most 20 files.
    """
    global _LOG_FH, _LOG_FILE
    _LOG_FH = None
    _LOG_FILE = ""
    try:
        base = os.environ.get("LOCALAPPDATA") or tempfile.gettempdir()
        log_dir = os.path.join(base, "esa-lookup", "logs")
        os.makedirs(log_dir, exist_ok=True)
        # Prune to the 20 most recent .log files.
        try:
            existing = sorted(
                (os.path.join(log_dir, n) for n in os.listdir(log_dir) if n.endswith(".log")),
                key=os.path.getmtime,
            )
            for old in existing[:-19]:
                try:
                    os.remove(old)
                except OSError:
                    pass
        except OSError:
            pass
        stamp = time.strftime("%Y%m%d-%H%M%S")
        path = os.path.join(log_dir, f"esa-lookup-{stamp}.log")
        _LOG_FH = open(path, "w", encoding="utf-8", buffering=1)
        _LOG_FILE = path
        return path
    except Exception:
        _LOG_FH = None
        _LOG_FILE = ""
        return ""


def _close_run_log() -> None:
    global _LOG_FH
    if _LOG_FH is not None:
        try:
            _LOG_FH.close()
        except Exception:
            pass
    _LOG_FH = None


def _file_log(level: str, msg: str) -> None:
    """Best-effort write to the current run's log file. Never raises."""
    if _LOG_FH is None:
        return
    try:
        _LOG_FH.write(
            f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {level.upper():5s} {msg}\n"
        )
    except Exception:
        pass


def _file_log_traceback() -> None:
    """Persist a full traceback of the current exception to the log file
    only -- the GUI shows the one-line summary, the file gets the full chain
    so a post-mortem can see what really happened."""
    if _LOG_FH is None:
        return
    try:
        _LOG_FH.write(traceback.format_exc())
        _LOG_FH.write("\n")
    except Exception:
        pass


def _log(on_event: EventFn, msg: str, level: str = "info") -> None:
    _file_log(level, msg)
    on_event("log", (msg, level))


def _status(on_event: EventFn, msg: str) -> None:
    _file_log("STAT", f"[status] {msg}")
    on_event("status", msg)


def _progress(on_event: EventFn, frac: float) -> None:
    # Progress ticks are too noisy for the file log.
    on_event("progress", max(0.0, min(1.0, frac)))


# ---------------------------------------------------------------------------
# Main pipeline
# ---------------------------------------------------------------------------

def _col_letter(idx: int) -> str:
    """1-based column index -> Excel letter (only up to ZZ, plenty here)."""
    result = ""
    n = idx
    while n > 0:
        n, r = divmod(n - 1, 26)
        result = chr(65 + r) + result
    return result


def _range(col_start: int, col_end: int, row_start: int, row_end: int) -> str:
    return (
        f"{_col_letter(col_start)}{row_start}:{_col_letter(col_end)}{row_end}"
    )


# Fix 3: SAP ALV exports typically use the ALV column TITLE, which is often a
# short description rather than the technical field name. Try the technical
# name first, then any known aliases. Extend per-site if needed.
SAP_COLUMN_ALIASES: dict[str, list[str]] = {
    "TANUM":       ["TANUM", "TO Number", "Transfer Order", "TrfOrd", "TrfOrdNo"],
    "ABLAD":       ["ABLAD", "Unloading Point", "UnloadPt"],
    "QMNUM":       ["QMNUM", "Notification", "Notification No", "Notification Number"],
    "RSNUM":       ["RSNUM", "Reservation", "Reservation No", "Reservation Number", "Res.Number"],
    "RSPOS":       ["RSPOS", "Item", "Item No", "Item Number", "Res.Item"],
    "OBJNR":       ["OBJNR", "Object Number", "Obj.Number", "Object No"],
    "DISP_MATNR":  ["DISP_MATNR", "Disp Material", "Disposition Material", "Disp.Material"],
    "DISP_QTY":    ["DISP_QTY", "Disp Qty", "Disposition Qty", "Disposition Quantity", "Disp.Qty"],
    "Z_SECTION":   ["Z_SECTION", "Section"],
    "Z_MODULE":    ["Z_MODULE", "Module"],
    "DESCRIPT":    ["DESCRIPT", "Description", "Descr."],
    "SALES_ORDER": ["SALES_ORDER", "Sales Order", "Sales Doc.", "Sales Doc", "Sales Doc. No."],
    "LID":         ["LID"],
}


def _resolve_column(df: pd.DataFrame, canonical: str) -> str | None:
    """Return whichever of `canonical`'s alias names is present in df, or None."""
    for name in SAP_COLUMN_ALIASES.get(canonical, [canonical]):
        if name in df.columns:
            return name
    return None


def _build_lookup(
    df: pd.DataFrame,
    key_cols: list[str],
    value_cols: list[str],
) -> tuple[dict, int, int]:
    """Return (lookup, dup_count, blank_key_count) built from df.

    Resolves each canonical SAP field name (e.g. "TANUM") to whichever alias
    (e.g. "TO Number") the ALV export actually used.

    Fix C: duplicate composite keys keep the FIRST occurrence and increment
    dup_count so the caller can warn -- overwriting silently loses data when
    a lookup table legitimately has multiple rows per key.

    Fix D: rows where ANY key part is blank are skipped, not just rows where
    ALL parts are blank. A composite of "12345|" would otherwise falsely
    match every SAP row with the same first part and a blank second.
    """
    needed = list(dict.fromkeys(key_cols + value_cols))
    resolved: dict[str, str] = {}
    missing: list[str] = []
    for c in needed:
        r = _resolve_column(df, c)
        if r is None:
            missing.append(c)
        else:
            resolved[c] = r
    if missing:
        raise RuntimeError(
            "SAP export is missing these expected columns: "
            f"{missing}\n"
            f"Columns present in the export: {list(df.columns)}\n"
            "Fix: edit the ALV layout in ZTBV so each missing field is shown "
            "(prefer 'Technical Name' as the column title) and re-save the "
            "default variant, OR add another alias to SAP_COLUMN_ALIASES in "
            "pipeline.py."
        )
    out: dict[str, dict] = {}
    dup_count = 0
    blank_key_count = 0
    for _, row in df.iterrows():
        parts = [normalize_key(row[resolved[c]]) for c in key_cols]
        if any(p == "" for p in parts):
            blank_key_count += 1
            continue
        composite = "|".join(parts)
        if composite in out:
            dup_count += 1
            continue  # keep first occurrence -- do not silently overwrite
        out[composite] = {
            c: ("" if pd.isna(row[resolved[c]]) else row[resolved[c]])
            for c in value_cols
        }
    return out, dup_count, blank_key_count


def _run_step(
    step: LookupStep,
    xl: ExcelCtx,
    sap: SapSession,
    tmp_dir: str,
    on_event: EventFn,
    step_index: int,
    total_steps: int,
    stop: threading.Event,
) -> tuple[int, int]:
    """Run one lookup step. Returns (matched, unmatched) row counts."""

    def sub_progress(sub_i: int, sub_n: int = 7):
        overall = (step_index + sub_i / sub_n) / total_steps
        _progress(on_event, overall)

    step_started = time.time()
    if stop.is_set():
        raise Cancelled()

    # --- 1. Read Excel keys in one COM call ------------------------------
    _status(on_event, f"[{step_index + 1}/{total_steps}] {step.name}: reading Excel keys")
    _log(on_event, f"--- Step {step_index + 1}/{total_steps}: {step.name}  "
                    f"(table={step.sap_table}, key_cols={step.key_columns})")
    sheet = xl.sheet

    # Fix F: each step derives its own last_row from its own primary key
    # column (matches notebook, which re-computes last_row per step). A short
    # column later in the workflow doesn't process phantom rows from a
    # longer column earlier in the workflow.
    primary_col = step.key_columns[0]
    last_row = last_row_in_column(sheet, primary_col)
    if last_row < 2:
        _log(on_event,
             f"No data below the header in column {_col_letter(primary_col)} "
             f"(#{primary_col}); step skipped", "warn")
        _progress(on_event, (step_index + 1) / total_steps)
        return (0, 0)

    key_col_min, key_col_max = min(step.key_columns), max(step.key_columns)
    key_range = _range(key_col_min, key_col_max, 2, last_row)
    key_rows = read_range_2d(sheet, key_range)
    sub_progress(1)

    first_key_offset = step.key_columns[0] - key_col_min

    def row_key(vals: list) -> str:
        offset_map = {c - key_col_min: c for c in step.key_columns}
        parts = [normalize_key(vals[offset]) for offset in sorted(offset_map)]
        return step.key_joiner.join(parts)

    # excel_keys is the normalized composite for every row -- used for both
    # lookup matching and (when match_key_column is set) the P-column audit
    # write. Notebook builds the identical string via `normalize_key(...)|
    # normalize_key(...)` inside its per-row loop.
    excel_keys = [row_key(r) for r in key_rows]
    # Skip flags: only used to (a) exclude from SAP paste, (b) count for the
    # log summary. The write-back loop treats skip rows as non-matches, which
    # is what the notebook does implicitly (its else branch fires for any
    # composite key that's not in the SAP result set, including "NOTFOUND|").
    skip_flags = [_is_skip_key_cell(r[first_key_offset]) for r in key_rows]

    unique_paste_values: list[str] = []
    seen: set[str] = set()
    for r, skipped in zip(key_rows, skip_flags):
        if skipped:
            continue
        # For paste, use the FIRST key column value (SAP filter dialog is
        # single-column). Multi-column keys still work because SAP returns
        # the full result set and we filter by composite key on our side.
        v = clean_numeric_for_sap(r[first_key_offset])
        if v and v not in seen:
            seen.add(v)
            unique_paste_values.append(v)

    n_skipped = sum(skip_flags)
    if n_skipped:
        _log(on_event,
             f"note: {n_skipped} row(s) in column {_col_letter(primary_col)} "
             f"are blank or 'NOT FOUND' -- not sent to SAP; their output "
             f"cells will be cleared just like any other non-match")

    if not unique_paste_values:
        _log(on_event,
             f"No usable values in column(s) {step.key_columns} "
             f"(after skipping blanks / 'NOT FOUND'); step is a no-op",
             "warn")
        _progress(on_event, (step_index + 1) / total_steps)
        return (0, 0)

    _log(on_event, f"{len(unique_paste_values)} unique key(s) will be sent to SAP")
    sub_progress(2)

    # --- 2. Stage clipboard, navigate SAP, paste filter, execute ---------
    scratch = stage_values_on_clipboard(xl.app, unique_paste_values)
    try:
        _status(on_event, f"[{step_index + 1}/{total_steps}] SAP: loading {step.sap_table}")
        open_ztbv_table(sap, step.sap_table, log=lambda m: _log(on_event, m))
        sub_progress(3)

        push_id = PUSH_BUTTONS[(step.sap_table, step.push_button_field)]
        _status(on_event, f"[{step_index + 1}/{total_steps}] SAP: pasting {len(unique_paste_values)} filter values")
        paste_multi_value_filter(sap, push_id, unique_paste_values, log=lambda m: _log(on_event, m))
        sub_progress(4)

        _status(on_event, f"[{step_index + 1}/{total_steps}] SAP: executing query")
        execute_query(sap, log=lambda m: _log(on_event, m))
        sub_progress(5)
    finally:
        close_scratch(scratch)

    if stop.is_set():
        raise Cancelled()

    # --- 3. Export ALV grid to a file -----------------------------------
    _status(on_event, f"[{step_index + 1}/{total_steps}] SAP: exporting ALV grid")
    ts = int(time.time())
    export_name = f"{step.sap_table}_{step.push_button_field}_{ts}.xlsx"
    export_start = time.time()
    export_path = export_alv_to_file(
        sap, tmp_dir, export_name, log=lambda m: _log(on_event, m)
    )
    try:
        size_kb = os.path.getsize(export_path) / 1024.0
    except OSError:
        size_kb = 0.0
    _log(on_event, f"export finished in {time.time() - export_start:.1f}s "
                    f"({size_kb:.0f} KB)")
    sub_progress(6)

    # --- 4. Load export, build lookup dict ------------------------------
    _status(on_event, f"[{step_index + 1}/{total_steps}] loading SAP export")
    df = pd.read_excel(export_path)
    _log(on_event, f"SAP returned {len(df)} row(s) with columns {list(df.columns)[:8]}{'...' if len(df.columns) > 8 else ''}")
    extra_sap_cols = [e.sap_col for e in step.extras if e.sap_col]
    lookup, dup_count, blank_key_count = _build_lookup(
        df,
        step.sap_key_columns,
        step.sap_output_columns + extra_sap_cols,
    )
    if dup_count:
        _log(on_event,
             f"WARNING: SAP returned {dup_count} duplicate key(s); only the "
             f"FIRST row per key is used. Check the run log file (path echoed "
             f"above) for the raw SAP columns, or tighten your ALV filter.",
             "warn")
    if blank_key_count:
        _log(on_event,
             f"note: skipped {blank_key_count} SAP row(s) with blank/partial "
             f"key columns", "info")

    # --- 5. Resolve each Excel row's fate: matched vs. else --------------
    # Skip rows use the same else branch as SAP non-matches -- the notebook
    # write loop makes no distinction (both fall through to the same clear
    # branch). See _SKIP_KEY_MARKERS docstring.
    _status(on_event, f"[{step_index + 1}/{total_steps}] matching keys and writing back to Excel")
    matched = 0
    entries_by_row: list[dict | None] = []
    for k, skipped in zip(excel_keys, skip_flags):
        if skipped:
            entries_by_row.append(None)
            continue
        entry = lookup.get(k)
        entries_by_row.append(entry)
        if entry is not None:
            matched += 1

    n_rows = len(excel_keys)
    rows_count = int(sheet.Rows.Count)

    # --- 6. Bulk write to Excel -----------------------------------------
    with bulk_write(xl.app):
        # -------- Main output block -------------------------------------
        oc_min = min(step.excel_output_columns)
        oc_max = max(step.excel_output_columns)
        oc_span = oc_max - oc_min + 1
        target_write = _range(oc_min, oc_max, 2, last_row)
        # Match notebook: clear the ENTIRE column range down to Rows.Count so
        # stale rows below last_row (from a prior, longer run) are wiped.
        target_clear = _range(oc_min, oc_max, 2, rows_count)

        main_out: list[list] = []
        for i in range(n_rows):
            entry = entries_by_row[i]
            row_vals = ["" for _ in range(oc_span)]
            if entry is not None:
                for j, sap_c in enumerate(step.sap_output_columns):
                    excel_c = step.excel_output_columns[j]
                    offset = excel_c - oc_min
                    val = entry[sap_c]
                    row_vals[offset] = "" if val is None else val
            main_out.append(row_vals)

        clear_range(sheet, target_clear)
        set_column_format_text(
            sheet, f"{_col_letter(oc_min)}:{_col_letter(oc_max)}"
        )
        write_range_2d(sheet, target_write, main_out)

        # -------- Extras (per-column, with per-column semantics) --------
        for extra in step.extras:
            letter = _col_letter(extra.excel_col)
            xrange = f"{letter}2:{letter}{last_row}"

            # Read existing only when at least one row will preserve it.
            need_existing = extra.preserve_on_nonmatch or extra.sap_col is None
            existing_extra = (
                read_range_2d(sheet, xrange) if need_existing else None
            )

            col_data: list[list] = []
            for i in range(n_rows):
                existing_val = (
                    existing_extra[i][0]
                    if (existing_extra and i < len(existing_extra)
                        and existing_extra[i])
                    else ""
                )
                entry = entries_by_row[i]
                if entry is not None:
                    # Matched row.
                    if extra.sap_col is None:
                        # No SAP source -> preserve on match
                        # (mirrors notebook TO-3 col J behavior).
                        col_data.append([existing_val])
                    else:
                        v = entry[extra.sap_col]
                        col_data.append(["" if v is None else v])
                else:
                    # Non-match OR skip (treated the same, per notebook).
                    if extra.preserve_on_nonmatch:
                        col_data.append([existing_val])
                    else:
                        col_data.append([""])

            # Only set text format on columns where we're writing SAP data;
            # a preserve-only column (sap_col=None) should keep whatever
            # NumberFormat the user had -- notebook doesn't touch it.
            if extra.sap_col is not None:
                set_column_format_text(sheet, f"{letter}:{letter}")
            write_range_2d(sheet, xrange, col_data)

        # -------- Match key column (P for TO-2) --------------------------
        if step.match_key_column is not None:
            mk_letter = _col_letter(step.match_key_column)
            # Header at row 1 + text format for the whole column, matching
            # notebook lines 851-852.
            sheet.Cells(1, step.match_key_column).Value = step.match_key_header
            set_column_format_text(sheet, f"{mk_letter}:{mk_letter}")
            # Notebook line 863 writes the composite key to P for every row
            # in the loop (including skip rows, which get "NOTFOUND|..." or
            # "|" written). Match that by writing excel_keys[i] for every
            # row 2..last_row.
            mk_range = f"{mk_letter}2:{mk_letter}{last_row}"
            mk_data = [[k] for k in excel_keys]
            write_range_2d(sheet, mk_range, mk_data)

    sub_progress(7)

    non_skip = n_rows - n_skipped
    unmatched = non_skip - matched
    # Sanity: if we sent >20 unique keys and SAP came back with <10% as many
    # rows as we asked about, something is off -- surface a warning so the
    # user does not silently accept mostly-empty output.
    if len(unique_paste_values) > 20 and len(lookup) < max(1, len(unique_paste_values) // 10):
        _log(on_event,
             f"WARNING: sent {len(unique_paste_values)} unique key(s) to SAP but "
             f"only {len(lookup)} matched. Common causes: (a) query returned an "
             f"error screen, (b) the ALV filter rejected the values, (c) plant "
             f"or table wrong for this environment.", "warn")
    _log(on_event,
         f"Step {step_index + 1} finished in {time.time() - step_started:.1f}s: "
         f"{matched}/{non_skip} rows matched, {unmatched} unmatched"
         + (f", {n_skipped} skipped" if n_skipped else ""),
         "ok")
    _progress(on_event, (step_index + 1) / total_steps)
    return matched, unmatched


@dataclass
class RunConfig:
    excel_path: str
    workflow: str            # "TO" or "NOTIF"
    stop_event: threading.Event


def run(cfg: RunConfig, on_event: EventFn) -> bool:
    """Entry point invoked on a background thread. Returns True on success."""
    # Fix H: initialize this worker thread's COM apartment. pywin32 does an
    # implicit CoInitialize on Dispatch, but the second Run click (new
    # worker thread, same process) can hit CO_E_NOTINITIALIZED on
    # GetActiveObject / GetObject("SAPGUI") without an explicit init here.
    # Uninitialize in finally so the thread exits clean.
    pythoncom.CoInitialize()
    run_started = time.time()
    log_path = _open_run_log()
    try:
        _log(on_event, f"esa-lookup starting workflow '{cfg.workflow}'", "info")
        if log_path:
            _log(on_event, f"detailed log file: {log_path}", "info")
        # Diagnostic: freeze the environment into the file so a post-mortem
        # can tell which Python / pandas / openpyxl the run used.
        try:
            import openpyxl as _openpyxl
            openpyxl_v = _openpyxl.__version__
        except Exception:
            openpyxl_v = "?"
        _file_log("info",
                  f"env: Python {sys.version.split()[0]} on "
                  f"{sys.platform}, pandas {pd.__version__}, openpyxl {openpyxl_v}, "
                  f"cwd={os.getcwd()}, excel={cfg.excel_path}")

        _status(on_event, "opening Excel")
        xl = excel_attach(cfg.excel_path)
        _log(on_event, f"attached to Excel: {os.path.basename(cfg.excel_path)}", "ok")

        _status(on_event, "attaching to SAP GUI")
        sap = sap_attach()
        _log(on_event, "attached to SAP GUI session", "ok")

        steps = WORKFLOWS[cfg.workflow]
        total_steps = len(steps)
        tmp_dir = os.path.join(tempfile.gettempdir(), "esa_lookup")

        # Pre-flight: verify the workflow's primary input column has data
        # before we even spin up SAP navigation. Later steps derive their
        # own last_row (Fix F) and skip themselves if their column is empty.
        primary_col = steps[0].key_columns[0]
        primary_last = last_row_in_column(xl.sheet, primary_col)
        if primary_last < 2:
            _log(on_event,
                 f"No data below the header in column "
                 f"{_col_letter(primary_col)} (#{primary_col}) -- the "
                 f"workflow's primary input. Aborting.", "error")
            on_event("done", False)
            return False
        _log(on_event,
             f"step 1 will process rows 2..{primary_last} (column "
             f"{_col_letter(primary_col)}); each subsequent step derives "
             f"its own row range from its own key column")

        totals_matched = 0
        totals_seen = 0
        for i, step in enumerate(steps):
            if cfg.stop_event.is_set():
                raise Cancelled()
            m, u = _run_step(
                step, xl, sap, tmp_dir, on_event, i, total_steps,
                cfg.stop_event,
            )
            totals_matched += m
            totals_seen += m + u

        # Fix L: save() now raises ExcelError on failure; warn the user
        # rather than silently pretending the write persisted.
        try:
            excel_save(xl.book)
        except ExcelError as e:
            _log(on_event, f"WARNING: {e}", "warn")

        _status(on_event, "done")
        _log(on_event, f"all steps complete: {totals_matched}/{totals_seen} row-matches across {total_steps} step(s)", "ok")
        _progress(on_event, 1.0)
        on_event("done", True)
        return True

    except Cancelled:
        _log(on_event, "cancelled by user", "warn")
        _status(on_event, "cancelled")
        on_event("done", False)
        return False
    except SapError as e:
        _log(on_event, f"SAP error: {e}", "error")
        _file_log_traceback()  # full chain into the log file for post-mortem
        _status(on_event, "SAP error")
        on_event("done", False)
        return False
    except ExcelError as e:
        _log(on_event, f"Excel error: {e}", "error")
        _file_log_traceback()
        _status(on_event, "Excel error")
        on_event("done", False)
        return False
    except Exception as e:
        _log(on_event, f"unexpected error: {e}", "error")
        _log(on_event, traceback.format_exc(), "error")
        _file_log_traceback()
        _status(on_event, "failed")
        on_event("done", False)
        return False
    finally:
        _log(on_event, f"total elapsed: {time.time() - run_started:.1f}s", "info")
        _close_run_log()
        try:
            pythoncom.CoUninitialize()
        except Exception:
            pass


## Notebook event handler

Replaces the .py app's tkinter callback with a plain `print()` so log
lines appear in cell output. Same `(kind, payload)` protocol the pipeline
emits.


In [ ]:
def print_event(kind: str, payload) -> None:
    """Simple stdout event handler for notebook use. Same (kind, payload)
    contract the .py app's tkinter callback receives."""
    if kind == "log":
        msg, level = payload
        prefix = {"ok": "[OK]  ", "warn": "[WARN]", "error": "[ERR] ",
                  "info": "      "}.get(level, "      ")
        print(f"{prefix} {msg}")
    elif kind == "status":
        print(f"...    {payload}")
    elif kind == "progress":
        pass  # too noisy for stdout
    elif kind == "done":
        ok = payload
        print(f"===== workflow {'succeeded' if ok else 'FAILED'} =====")


---

## Configure your run

Set `WORKFLOW` below (`"TO"` or `"NOTIF"`), then run the cell. A native
Windows file-picker dialog pops up so you can browse to the Excel workbook.
Cancel the dialog to abort.

> If the dialog does not appear on top, look for it in the taskbar — some
> Jupyter front-ends push new native windows behind the browser.


In [ ]:
# ---- EDIT THIS -----------------------------------------------------------
WORKFLOW = "TO"   # "TO" or "NOTIF"
# --------------------------------------------------------------------------

# Native file-picker (same widget the .py GUI uses, via tkinter). Skip
# manual path-editing -- browse to the workbook you want to process.
import tkinter as tk
from tkinter import filedialog

_picker_root = tk.Tk()
_picker_root.withdraw()
_picker_root.attributes("-topmost", True)
try:
    EXCEL_PATH = filedialog.askopenfilename(
        title="Select the Excel workbook to process",
        filetypes=[("Excel workbooks", "*.xlsx *.xlsm *.xlsb"),
                   ("All files", "*.*")],
    )
finally:
    _picker_root.destroy()

if not EXCEL_PATH:
    raise SystemExit("No file selected -- aborting. Re-run this cell to retry.")

if WORKFLOW not in ("TO", "NOTIF"):
    raise SystemExit(f"WORKFLOW must be 'TO' or 'NOTIF', got {WORKFLOW!r}")

print(f"Workflow:    {WORKFLOW}")
print(f"Excel file:  {EXCEL_PATH}")


## Execute

Runs the full workflow against the file you just picked. Log lines appear
below as each step progresses; the final `===== workflow succeeded =====`
banner means the workbook has been saved.


In [ ]:
cfg = RunConfig(
    excel_path=EXCEL_PATH,
    workflow=WORKFLOW,
    stop_event=threading.Event(),
)
run(cfg, on_event=print_event)
